# Final train-validation run

Use this notebook only after choosing a configuration from validation results. It supports a manual frozen configuration or a persisted grid-search winner. It does not load the test manifest.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json

import matplotlib.pyplot as plt
import torch
from torch.utils.data import Subset

from reservoir_segmentation.training.config import ExperimentConfig, RuntimeConfig
from reservoir_segmentation.training.data import (
    build_dataloader, build_split_datasets, fit_channel_statistics, select_overfit_indices, write_channel_statistics,
)
from reservoir_segmentation.training.engine import overfit_smoke_test, run_training


In [ ]:
REPOSITORY_ROOT = Path.cwd().resolve()
GEOMETRY_ROOT = REPOSITORY_ROOT / 'data/processed/riverscope/geometry-v2'
SOURCE_PATH = REPOSITORY_ROOT / 'data/raw/RiverScope.zip'
RUN_ROOT = REPOSITORY_ROOT / 'artifacts/experiments/final_train_validation'
FULL_TRAIN_DEVICE = 'cuda'
SMOKE_DEVICE = 'cpu'

if FULL_TRAIN_DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Full training requires CUDA. Use CPU only for tests and the smoke test.')

raw_datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH)
statistics = fit_channel_statistics(raw_datasets['train'])
RUN_ROOT.mkdir(parents=True, exist_ok=True)
write_channel_statistics(statistics, RUN_ROOT / 'train_channel_statistics.json')
datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH, statistics=statistics)


## Choose a frozen validation configuration

Set `USE_GRID_WINNER` to load a selected persisted winner, or replace `manual_config` with explicitly reviewed values. Do not use test results to make this choice.

In [ ]:
USE_GRID_WINNER = True
WINNER_PATH = REPOSITORY_ROOT / 'artifacts/experiments/grid_search/weighted/winner.json'
manual_config = ExperimentConfig(runtime=RuntimeConfig(seed=42, device=FULL_TRAIN_DEVICE, num_workers=2))

if USE_GRID_WINNER:
    winner_payload = json.loads(WINNER_PATH.read_text())
    config = ExperimentConfig.from_dict(winner_payload['experiment_config'])
    config = replace(config, runtime=replace(config.runtime, device=FULL_TRAIN_DEVICE, num_workers=2))
else:
    config = manual_config

config.to_dict()


## Required overfit smoke test

This repeats one deterministic train batch only. Its loss should fall substantially; investigate data alignment, masks, and gradients before starting the full run if it does not. It is not used for model selection.

In [ ]:
smoke_indices = select_overfit_indices(datasets['train'], count=1)
smoke_loader = build_dataloader(
    Subset(datasets['train'], smoke_indices), batch_size=1, shuffle=False,
    seed=config.runtime.seed, num_workers=config.runtime.num_workers,
)
SMOKE_STEPS = 50
smoke_config = replace(config, runtime=replace(config.runtime, device=SMOKE_DEVICE))
smoke = overfit_smoke_test(smoke_loader, smoke_config, steps=SMOKE_STEPS)
smoke


In [ ]:
if not smoke['final_loss'] < smoke['initial_loss']:
    raise RuntimeError('Overfit smoke test did not reduce loss; do not start full training.')

train_loader = build_dataloader(
    datasets['train'], batch_size=config.optimization.batch_size, shuffle=True,
    seed=config.runtime.seed, num_workers=config.runtime.num_workers,
)
validation_loader = build_dataloader(
    datasets['valid'], batch_size=config.optimization.batch_size, shuffle=False,
    seed=config.runtime.seed, num_workers=config.runtime.num_workers,
)
result = run_training(
    train_loader, validation_loader, config, output_dir=RUN_ROOT,
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
)
result['summary']


## Curves and frozen checkpoint

The trainer has already restored its selected validation checkpoint. Save the curves and checkpoint path for the final test notebook; do not tune based on test observations.

In [ ]:
history = result['history']
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs, [row['training']['loss'] for row in history], label='train')
axes[0].plot(epochs, [row['validation']['loss'] for row in history], label='validation')
axes[0].set(title='Masked loss', xlabel='Epoch'); axes[0].legend()
for metric in ('dice', 'iou', 'masked_accuracy'):
    axes[1].plot(epochs, [row['training']['regional'][metric] for row in history], label=f'train {metric}')
    axes[1].plot(epochs, [row['validation']['regional'][metric] for row in history], '--', label=f'validation {metric}')
axes[1].set(title='Regional metrics', xlabel='Epoch'); axes[1].legend(fontsize=7)
axes[2].plot(epochs, [row['validation']['shoreline']['selection_assd_macro_px'] for row in history])
axes[2].set(title='Validation selection ASSD', xlabel='Epoch', ylabel='Pixels')
fig.tight_layout(); fig.savefig(RUN_ROOT / 'train_validation_curves.png', dpi=160)
selected_checkpoint = result['best_checkpoint']
(RUN_ROOT / 'selected_checkpoint.txt').write_text(str(selected_checkpoint.resolve()) + '\n')
selected_checkpoint
